# 14. Multi-Domain Features + Bi-LSTM + Graph Attention for Spectrum Prediction

Adapts the framework from the paper:
**"Multi-domain Features Enhanced Time Series Prediction Framework Based on Bi-Directional LSTM and Graph Attention Network"** (IEEE DLCV 2025).

## Paper (EEG seizure prediction) → Our setting (spectrum AU% prediction)
- **Paper:** Multi-channel EEG → time-domain stats, frequency-domain (FFT/Welch), complex-network features → multi-head attention fusion → GAT (spatial) + Bi-LSTM (temporal) → binary classification.
- **Here:** Same data as notebooks 10–13 (72h→24h). We treat the 72h window as **6 segments** (12h each); each segment = one **node** with **time-domain** (mean, std, max, min) and **frequency-domain** (FFT power) features. **Adjacency** = adjacent segments. **GAT** over 6 nodes captures segment relations; **Bi-LSTM** on raw (72,1) captures temporal dynamics; **fusion** → Dense(24) for next-day regression.

## Same setup as 10–13
- Data: `work_dir/final`, LOOKBACK=72, FORECAST_HORIZON=24. Metrics: MAE, RMSE, MASE. Visuals: results table, bar charts, improvement %, predicted vs actual (dashed), mean profile, per-hour MAE, residuals, insights, one-line summary.


In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

print(f'TensorFlow: {tf.__version__}')


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_visible_devices(gpus, 'GPU')
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'GPU enabled: {len(gpus)} device(s)')
        with tf.device('/GPU:0'):
            _ = tf.constant(1)
        print('GPU ready.')
    except RuntimeError as e:
        print('GPU config:', e)
else:
    print('No GPU. On Apple Silicon: pip install tensorflow-metal')
USE_GPU = len(gpus) > 0
num_cores = os.cpu_count() or 4
tf.config.threading.set_intra_op_parallelism_threads(num_cores)
tf.config.threading.set_inter_op_parallelism_threads(num_cores)


## Data loading (same as notebook 10)

In [ ]:
work_dir = Path("work_dir")
if not work_dir.exists():
    work_dir = Path("../work_dir")
final_dir = work_dir / "final"
training_dir = final_dir / "training"
testing_dir = final_dir / "testing"
if not final_dir.exists() or not training_dir.exists() or not testing_dir.exists():
    raise FileNotFoundError("work_dir/final/training and testing not found")

class_options = sorted([d.name for d in training_dir.iterdir() if d.is_dir()])
LOOKBACK = 72
FORECAST_HORIZON = 24
print(f"Bands: {class_options}, Lookback={LOOKBACK}, Horizon={FORECAST_HORIZON}")

def load_data_for_band(band_name: str, split: str):
    split_dir = final_dir / split / band_name
    if not split_dir.exists():
        return pd.DataFrame()
    dfs = []
    for p in sorted(split_dir.glob("final_*.parquet")):
        try:
            dfs.append(pd.read_parquet(p))
        except Exception as e:
            print(f"Error loading {p}: {e}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def prepare_next_day_sequences(train_df, test_df, lookback=72, forecast_horizon=24):
    freqs = sorted(train_df["freq_center_ghz"].unique())
    ths = sorted(train_df["threshold_dbm"].unique())
    if len(ths) > 1:
        train_df = train_df[train_df["threshold_dbm"] == ths[0]].copy()
        test_df = test_df[test_df["threshold_dbm"] == ths[0]].copy()
    X_tr, y_tr, X_te, y_te = [], [], [], []
    for freq in freqs:
        tr = train_df[train_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        te = test_df[test_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        for i in range(len(tr) - lookback - forecast_horizon + 1):
            X_tr.append(tr[i:i+lookback])
            y_tr.append(tr[i+lookback:i+lookback+forecast_horizon])
        n_test_days = len(te) // forecast_horizon
        for d in range(n_test_days):
            if d == 0:
                inp = tr[-lookback:] if len(tr) >= lookback else np.concatenate([np.zeros(lookback - len(tr)), tr])
            else:
                h = max(0, lookback - d * forecast_horizon)
                inp = np.concatenate([tr[-h:], te[0:d*forecast_horizon]]) if h > 0 else te[d*forecast_horizon - lookback:d*forecast_horizon]
            tgt = te[d*forecast_horizon:(d+1)*forecast_horizon]
            if len(inp) == lookback and len(tgt) == forecast_horizon:
                X_te.append(inp)
                y_te.append(tgt)
    if not X_tr or not X_te:
        return np.array([]), np.array([]), np.array([]), np.array([])
    return np.array(X_tr).reshape(-1, lookback, 1), np.array(y_tr), np.array(X_te).reshape(-1, lookback, 1), np.array(y_te)

In [ ]:
train_data_by_band = {}
test_data_by_band = {}
for band in class_options:
    tr = load_data_for_band(band, "training")
    te = load_data_for_band(band, "testing")
    if not tr.empty and not te.empty:
        train_data_by_band[band] = tr
        test_data_by_band[band] = te
X_train_list, y_train_list, X_test_list, y_test_list = [], [], [], []
for band in class_options:
    if band not in train_data_by_band:
        continue
    X_tr, y_tr, X_te, y_te = prepare_next_day_sequences(train_data_by_band[band], test_data_by_band[band], LOOKBACK, FORECAST_HORIZON)
    if len(X_tr) > 0 and len(X_te) > 0:
        X_train_list.append(X_tr)
        y_train_list.append(y_tr)
        X_test_list.append(X_te)
        y_test_list.append(y_te)
X_train = np.vstack(X_train_list)
y_train = np.vstack(y_train_list)
X_test = np.vstack(X_test_list)
y_test = np.vstack(y_test_list)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
scaler_X = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))
X_train_scaled = scaler_X.fit_transform(X_train.reshape(-1, 1)).reshape(X_train.shape)
X_test_scaled = scaler_X.transform(X_test.reshape(-1, 1)).reshape(X_test.shape)
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).reshape(y_train.shape)
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).reshape(y_test.shape)
print("Data normalized.")

## Multi-domain segment features (paper: time + frequency + attention)

Split 72h into **6 segments** of 12h. Per segment: **time-domain** (mean, std, max, min), **frequency-domain** (FFT power over 12 points). Result: (n_samples, 6, n_feat). Adjacency: segment i connected to i±1.

In [ ]:
N_SEGMENTS = 6
SEG_LEN = LOOKBACK // N_SEGMENTS  # 12

def extract_segment_features(x: np.ndarray) -> np.ndarray:
    """x: (batch, 72, 1). Out: (batch, N_SEGMENTS, n_feat). Time + freq per segment."""
    batch = x.shape[0]
    feats = []
    for s in range(N_SEGMENTS):
        seg = x[:, s*SEG_LEN:(s+1)*SEG_LEN, 0]  # (batch, 12)
        t_mean = np.mean(seg, axis=1, keepdims=True)
        t_std = np.std(seg, axis=1, keepdims=True) + 1e-8
        t_max = np.max(seg, axis=1, keepdims=True)
        t_min = np.min(seg, axis=1, keepdims=True)
        fft_seg = np.fft.rfft(seg, axis=1)
        power = np.abs(fft_seg) ** 2
        f_mean = np.mean(power, axis=1, keepdims=True)
        f_max = np.max(power, axis=1, keepdims=True)
        feats.append(np.hstack([t_mean, t_std, t_max, t_min, f_mean, f_max]))
    out = np.stack(feats, axis=1)  # (batch, 6, 6)
    return out.astype(np.float32)

X_train_node = extract_segment_features(X_train_scaled)
X_test_node = extract_segment_features(X_test_scaled)
node_scaler = MinMaxScaler(feature_range=(0, 1))
B, N, F = X_train_node.shape
X_train_node = node_scaler.fit_transform(X_train_node.reshape(-1, F)).reshape(B, N, F)
B, N, F = X_test_node.shape
X_test_node = node_scaler.transform(X_test_node.reshape(-1, F)).reshape(B, N, F)
print("Node features shape:", X_train_node.shape)

# Adjacency: segment i adjacent to i-1, i+1
ADJ = np.zeros((N_SEGMENTS, N_SEGMENTS))
for i in range(N_SEGMENTS):
    if i > 0:
        ADJ[i, i-1] = 1
    if i < N_SEGMENTS - 1:
        ADJ[i, i+1] = 1
    ADJ[i, i] = 1
print("Adjacency (segment graph):\n", ADJ)

## GAT (graph attention) + Bi-LSTM fusion model (paper: spatial + temporal)

- **GAT:** One-head attention over neighbors (paper Eq. 5–8). Input node features (batch, 6, 6) → output (batch, 6, F_out) → mean pool → (batch, F_out).
- **Bi-LSTM:** On raw sequence (72, 1) → last state → (batch, 2*units).
- **Fusion:** Concat [GAT_vec, BiLSTM_vec] → Dense(24).

In [ ]:
class GATLayer(keras.layers.Layer):
    """Single-head graph attention. Input H (B,N,F), adj (N,N). Output (B,N,F_out)."""
    def __init__(self, F_out: int, adj: np.ndarray, **kwargs):
        super().__init__(**kwargs)
        self.F_out = F_out
        self.adj = tf.constant(adj, dtype=tf.float32)
        self.N = adj.shape[0]

    def build(self, input_shape):
        F_in = int(input_shape[-1])
        self.W = self.add_weight("W", shape=[F_in, self.F_out], initializer="glorot_uniform")
        self.a = self.add_weight("a", shape=[2 * self.F_out, 1], initializer="glorot_uniform")
        super().build(input_shape)

    def call(self, inputs):
        H = inputs  # (B, N, F_in)
        W_h = tf.matmul(H, self.W)  # (B, N, F_out)
        W_i = tf.expand_dims(W_h, 2)
        W_j = tf.expand_dims(W_h, 1)
        W_i_rep = tf.tile(W_i, [1, 1, self.N, 1])
        W_j_rep = tf.tile(W_j, [1, self.N, 1, 1])
        concat = tf.concat([W_i_rep, W_j_rep], axis=-1)
        e = tf.nn.leaky_relu(tf.matmul(concat, self.a))
        e = tf.squeeze(e, -1)
        mask = (1 - self.adj) * -1e9
        e = e + mask
        alpha = tf.nn.softmax(e, axis=-1)
        H_new = tf.matmul(alpha, W_h)
        return tf.nn.elu(H_new)

def build_gat_bilstm_model(lookback, horizon, n_nodes, n_node_feat, adj, gat_units=16, lstm_units=16):
    # Branch 1: GAT on segment nodes
    node_in = keras.Input(shape=(n_nodes, n_node_feat), name="node_input")
    gat = GATLayer(gat_units, adj, name="gat")(node_in)
    gat_pool = layers.GlobalAveragePooling1D()(gat)
    # Branch 2: Bi-LSTM on raw sequence
    seq_in = keras.Input(shape=(lookback, 1), name="seq_input")
    bilstm = layers.Bidirectional(layers.LSTM(lstm_units, activation="tanh"))(seq_in)
    # Fusion
    fused = layers.Concatenate()([gat_pool, bilstm])
    out = layers.Dense(32, activation="relu")(fused)
    out = layers.Dense(horizon, activation="linear")(out)
    model = keras.Model(inputs=[node_in, seq_in], outputs=out)
    return model

In [ ]:
model = build_gat_bilstm_model(LOOKBACK, FORECAST_HORIZON, N_SEGMENTS, X_train_node.shape[-1], ADJ, gat_units=16, lstm_units=16)
model.compile(optimizer=keras.optimizers.Adam(0.001), loss="mse", metrics=["mae"])
EPOCHS = 50
BATCH = 128 if USE_GPU else 32
model.fit(
    [X_train_node, X_train_scaled],
    y_train_scaled,
    epochs=EPOCHS,
    batch_size=BATCH,
    validation_split=0.2,
    verbose=1,
)
y_pred_gat_bilstm_scaled = model.predict([X_test_node, X_test_scaled], verbose=0)
y_pred_gat_bilstm = scaler_y.inverse_transform(y_pred_gat_bilstm_scaled.reshape(-1, 1)).reshape(y_test.shape)
print("GAT+Bi-LSTM trained and predicted.")

In [ ]:
# Conventional Bi-LSTM only (no GAT, no multi-domain) for comparison
model_bilstm = keras.Sequential([
    layers.Bidirectional(layers.LSTM(16, activation="tanh"), input_shape=(LOOKBACK, 1)),
    layers.Dense(32, activation="relu"),
    layers.Dense(FORECAST_HORIZON, activation="linear"),
])
model_bilstm.compile(optimizer=keras.optimizers.Adam(0.001), loss="mse", metrics=["mae"])
model_bilstm.fit(X_train_scaled, y_train_scaled, epochs=EPOCHS, batch_size=BATCH, validation_split=0.2, verbose=1)
y_pred_bilstm_scaled = model_bilstm.predict(X_test_scaled, verbose=0)
y_pred_bilstm = scaler_y.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1)).reshape(y_test.shape)

def naive_predictor(X, horizon):
    last = X[:, -1, 0]
    return np.tile(last.reshape(-1, 1), (1, horizon))
y_pred_naive = naive_predictor(X_test, FORECAST_HORIZON)

def calculate_mae(y_true, y_pred):
    return mean_absolute_error(y_true.flatten(), y_pred.flatten())
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
def calculate_mase(y_true, y_pred, y_train):
    mae = np.mean(np.abs(y_true.flatten() - y_pred.flatten()))
    scale = np.mean(np.abs(np.diff(y_train.flatten()))) if len(y_train.flatten()) > 1 else 1.0
    return mae / max(scale, 1e-8)

In [ ]:
# Results table (same format as notebook 10)
mae_gat = calculate_mae(y_test, y_pred_gat_bilstm)
rmse_gat = calculate_rmse(y_test, y_pred_gat_bilstm)
mase_gat = calculate_mase(y_test, y_pred_gat_bilstm, y_train)
mae_bi = calculate_mae(y_test, y_pred_bilstm)
rmse_bi = calculate_rmse(y_test, y_pred_bilstm)
mase_bi = calculate_mase(y_test, y_pred_bilstm, y_train)
mae_naive = calculate_mae(y_test, y_pred_naive)
rmse_naive = calculate_rmse(y_test, y_pred_naive)
mase_naive = calculate_mase(y_test, y_pred_naive, y_train)

results_df = pd.DataFrame([
    {"Model": "GAT+Bi-LSTM (multi-domain)", "MAE": mae_gat, "RMSE": rmse_gat, "MASE": mase_gat},
    {"Model": "Bi-LSTM only", "MAE": mae_bi, "RMSE": rmse_bi, "MASE": mase_bi},
    {"Model": "Naive Baseline", "MAE": mae_naive, "RMSE": rmse_naive, "MASE": mase_naive},
])
results_df["MAE"] = results_df["MAE"].round(4)
results_df["RMSE"] = results_df["RMSE"].round(4)
results_df["MASE"] = results_df["MASE"].round(4)

print("\n" + "="*80)
print("FINAL RESULTS SUMMARY (same format as notebook 10)")
print("="*80)
print(f"Lookback: {LOOKBACK}h, Forecast: {FORECAST_HORIZON}h")
print(f"Test samples: {len(y_test)}")
print(f"\n{results_df.to_string(index=False)}")
display(results_df)

## Analysis and Visualizations (same as notebook 10)

In [ ]:
# 1. Bar charts: MAE, RMSE, MASE
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['MAE', 'RMSE', 'MASE']
colors = ['#2ecc71', '#3498db', '#9b59b6']
for ax, metric, color in zip(axes, metrics, colors):
    vals = results_df[metric].values
    bars = ax.bar(results_df['Model'], vals, color=color, edgecolor='black', linewidth=0.5)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} by Model')
    ax.tick_params(axis='x', rotation=15)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02 * max(vals), f'{v:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.suptitle('Next-day prediction: metric comparison', y=1.02, fontsize=12)
plt.show()

In [ ]:
# 2. Improvement over Naive Baseline (%)
naive_mae = results_df[results_df['Model'] == 'Naive Baseline']['MAE'].values[0]
naive_rmse = results_df[results_df['Model'] == 'Naive Baseline']['RMSE'].values[0]
naive_mase = results_df[results_df['Model'] == 'Naive Baseline']['MASE'].values[0]
improvement = results_df[results_df['Model'] != 'Naive Baseline'].copy()
improvement['MAE_imp_%'] = (1 - improvement['MAE'] / naive_mae) * 100
improvement['RMSE_imp_%'] = (1 - improvement['RMSE'] / naive_rmse) * 100
improvement['MASE_imp_%'] = (1 - improvement['MASE'] / naive_mase) * 100
print('Improvement over Naive Baseline (%):')
display(improvement[['Model', 'MAE_imp_%', 'RMSE_imp_%', 'MASE_imp_%']].round(2))
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(improvement))
w = 0.25
ax.bar(x - w, improvement['MAE_imp_%'], w, label='MAE', color='#2ecc71')
ax.bar(x, improvement['RMSE_imp_%'], w, label='RMSE', color='#3498db')
ax.bar(x + w, improvement['MASE_imp_%'], w, label='MASE', color='#9b59b6')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(improvement['Model'], rotation=15)
ax.set_ylabel('Improvement (%)')
ax.legend()
ax.set_title('Improvement over Naive Baseline (positive = better)')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Predicted vs Actual (up to 3 samples, actual dashed) + mean profile
hours = np.arange(FORECAST_HORIZON)
n_show = min(3, len(y_test))
fig, axes = plt.subplots(n_show, 1, figsize=(12, 4*n_show))
if n_show == 1:
    axes = [axes]
for i in range(n_show):
    ax = axes[i]
    ax.plot(hours, y_test[i], 'k--', linewidth=2.5, label='Actual', alpha=0.8)
    ax.plot(hours, y_pred_gat_bilstm[i], '-', linewidth=1.6, label='GAT+Bi-LSTM')
    ax.plot(hours, y_pred_bilstm[i], '-', linewidth=1.2, label='Bi-LSTM only', alpha=0.8)
    ax.plot(hours, y_pred_naive[i], '-', linewidth=1.2, label='Naive', alpha=0.7)
    ax.set_title(f'Test sample {i+1}: Predicted (solid) vs Actual (dashed)')
    ax.set_xlabel('Hour')
    ax.set_ylabel('AU (%)')
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hours, y_test.mean(axis=0), 'k--', linewidth=2.5, label='Actual (mean)')
ax.plot(hours, y_pred_gat_bilstm.mean(axis=0), '-', linewidth=1.6, label='GAT+Bi-LSTM (mean)')
ax.plot(hours, y_pred_bilstm.mean(axis=0), '-', linewidth=1.2, label='Bi-LSTM (mean)', alpha=0.8)
ax.plot(hours, y_pred_naive.mean(axis=0), '-', linewidth=1.2, label='Naive (mean)', alpha=0.7)
ax.set_title('Mean 24h profile: predicted vs actual')
ax.set_xlabel('Hour')
ax.set_ylabel('AU (%)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 4. Per-hour MAE
mae_per_hour_gat = np.abs(y_test - y_pred_gat_bilstm).mean(axis=0)
mae_per_hour_bi = np.abs(y_test - y_pred_bilstm).mean(axis=0)
mae_per_hour_naive = np.abs(y_test - y_pred_naive).mean(axis=0)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hours, mae_per_hour_gat, '-o', label='GAT+Bi-LSTM', markersize=4)
ax.plot(hours, mae_per_hour_bi, '-o', label='Bi-LSTM only', markersize=4, alpha=0.8)
ax.plot(hours, mae_per_hour_naive, '-o', label='Naive', markersize=4, alpha=0.7)
ax.set_title('MAE by forecast hour')
ax.set_xlabel('Hour')
ax.set_ylabel('MAE (%)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 5. Residuals: GAT+Bi-LSTM vs Naive
residuals_best = (y_test - y_pred_gat_bilstm).flatten()
residuals_naive = (y_test - y_pred_naive).flatten()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(residuals_best, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('Residual (actual - predicted)')
axes[0].set_ylabel('Count')
axes[0].set_title('Residuals: GAT+Bi-LSTM')
axes[1].hist(residuals_naive, bins=50, color='#95a5a6', edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (actual - predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals: Naive Baseline')
plt.suptitle('Residual distribution (centered at 0 is ideal)', y=1.02)
plt.tight_layout()
plt.show()
print(f'Residual mean (bias): GAT+Bi-LSTM = {residuals_best.mean():.4f}, Naive = {residuals_naive.mean():.4f}')
print(f'Residual std:         GAT+Bi-LSTM = {residuals_best.std():.4f}, Naive = {residuals_naive.std():.4f}')

### Key insights

- **GAT+Bi-LSTM (paper):** Multi-domain segment features (time + frequency) plus graph attention over segments capture structure; Bi-LSTM captures temporal dynamics. Fusion improves over Bi-LSTM alone when segment relations matter.
- **Improvement over naive:** Positive % means the model beats the last-value baseline.
- **Mean profile / per-hour MAE / residuals:** Same interpretation as notebooks 10–13.

In [ ]:
# One-line summary (same format as notebook 10)
best_row = results_df[results_df['Model'] == 'GAT+Bi-LSTM (multi-domain)'].iloc[0]
imp_mae = (1 - best_row['MAE'] / naive_mae) * 100
print(f"Best model: GAT+Bi-LSTM (MAE={best_row['MAE']:.4f}, RMSE={best_row['RMSE']:.4f}, MASE={best_row['MASE']:.4f}).")
print(f"Improvement over Naive: MAE {imp_mae:+.1f}%.")